In [7]:
from git import Repo
from langchain_text_splitters import Language
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain


In [1]:
print("ok")

ok


In [3]:
%pwd

'e:\\GenAi\\Project\\End-To-End-Source-Code-Analysis'

In [2]:
import os
os.chdir("../")

In [23]:
import os
os.chdir("Scode_download_repo")



In [22]:
!mkdir Scode_download_repo 

In [8]:
repo_path = "E:\\GenAi\\Project\\Scode_download_repo"
repo = Repo.clone_from("https://github.com/Gunavant-07/Medical-Chat-Boat.git",to_path=repo_path)


In [11]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob="**/*",
                                       suffixes=[".py"],
                                       parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

In [12]:
documents = loader.load()

In [14]:
len(documents)

7

In [15]:
document_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, chunk_size=500, chunk_overlap=20)

In [16]:
text = document_splitter.split_documents(documents)

In [18]:
len(text)

14

In [19]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [20]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings


In [24]:
def download_huggingface_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [25]:
embeddings = download_huggingface_embeddings() 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1540.79it/s]


In [26]:
Chromadb = Chroma.from_documents(text, embedding=embeddings, persist_directory="E:\\GenAi\\Project\\End-To-End-Source-Code-Analysis\\research\\db")

In [27]:
Chromadb

In [30]:
llm = ChatOpenAI(
    model="gpt-5.6-luna",
    api_key=os.getenv("Openai_API_KEY"),
    temperature=0.4,
)

In [31]:
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

C:\Users\desai\AppData\Local\Temp\ipykernel_23744\1764516168.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [32]:
qa = ConversationalRetrievalChain.from_llm(llm, retriever=Chromadb.as_retriever(search_type="mmr", search_kwargs={"k": 8}), memory = memory)

In [35]:
question = "what is load pdf file function in the code"

In [36]:
result = qa({"question": question})
print(result['answer'])

The `load_pdf_file()` function loads all PDF files from a specified directory.

It uses `DirectoryLoader` with `PyPDFLoader` to read and parse the PDFs, then returns their contents as a collection of documents:

```python
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents
```
